In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import chronopack as cp


# Load preprocessing checkpoint

In [2]:
adata_full = sc.read_h5ad("../output/adata_full.chk0.h5ad")
adata_hvg = sc.read_h5ad("../output/adata_hvg.chk0.h5ad")
adata_hvg.obs["time"] = pd.to_numeric(adata_hvg.obs["time"].astype(str))
adata_hvg.X = adata_hvg.layers["corrected"].copy()


## Waddington-OT (Transport Maps)

Compute WOT transition maps from the preprocessed, corrected AnnData checkpoint.

In [ ]:
wot_out = cp.tl.get_trajectories(
    adata_hvg,
    tmap_dir="tmaps_sc/",
    ot_model_kwargs=dict(epsilon=0.05, lambda1=1, lambda2=50, growth_iters=3, day_field="time"),
    compute_transport_maps=True,
    transition_topk=30,
    symmetrize="average",
    use_unpruned_as_pruned=True,
    graph_key="ot_graph",
    umap_key="ot_umap",
    umap_random_state=10,
)

tmaps = wot_out["tmaps"]
transitions_long = wot_out["transitions_long"]
tmats = wot_out["tmats"]
T_full = wot_out["T_full"]
T_pruned = wot_out["T_pruned"]
T_sym = wot_out["T_sym"]
adata_hvg.obsp["t_pruned"] = T_pruned
key_out = "ot_umap"
adata_hvg.obsm[key_out][:, 0] *= -1


# Cluster trajectories

In [ ]:
S = cp.tl.cluster_cells_to_trajectories(
    adata_hvg,
    T_pruned,
    cluster_key="traj_clusters",
    resolution=0.5,
    terminal_topk=200,
    graph_topk=500,
    random_state=0,
)
sc.pl.embedding(
    adata_hvg,
    basis=key_out,
    color=["time", "traj_clusters", "total_counts", "n_genes_by_counts", "NFKBIA", "HBE1", "GYPA"],
    ncols=3,
    cmap="viridis",
    size=25,
)


# Calculate terminal fates & trajectory cluster mass retention

In [ ]:
F, fate_names = cp.tl.compute_terminal_fate_probabilities(
    adata_hvg,
    terminal_cluster_key="traj_clusters",
    out_key="terminal_fate_probs_traj",
)
cp.pl.cell_trajectories.fate_probabilities(
    adata_hvg,
    fate_key="terminal_fate_probs_traj",
    fate_names_key="terminal_fate_probs_traj_fates",
    embedding_basis=key_out,
    size=25,
)


In [ ]:
out = cp.pl.cell_trajectories.mass_retention(
    adata_hvg,
    S=T_pruned,
    cluster_key="traj_clusters",
    n_reps=50,
    clusters_to_plot=["0", "1"],
    embedding_basis=key_out,
    size=25,
)
cp.pl.cell_trajectories.proportions_over_time(adata_hvg, cluster_key="traj_clusters")


# Save trajectory outputs

In [7]:
adata_full_out = adata_full[adata_hvg.obs_names, :].copy()
adata_full_out.obs = adata_hvg.obs.copy()
adata_full_out.obsm = adata_hvg.obsm.copy()
adata_full_out.obsp = adata_hvg.obsp.copy()
adata_full_out.uns = adata_hvg.uns.copy()

for _adata_to_write in (adata_full_out, adata_hvg):
    for _key, _value in list(_adata_to_write.uns.items()):
        if isinstance(_value, pd.Series):
            _adata_to_write.uns[_key] = _value.to_frame("value")

adata_full_out.write_h5ad("../output/adata_full.h5ad")
adata_hvg.write_h5ad("../output/adata_hvg.h5ad")
